# Model A: Behavioral Anomaly Detection (Isolation Forest)

In this notebook, we ingest the real `TS-PS4-1.csv` dataset (50,000+ records) to detect **Dormant Funds** anomalies.

Before feeding data into the model, we perform **Rigorous Data Preprocessing** (handling nulls, type-casting, and standard scaling). We then engineer features per beneficiary and isolate those who receive massive amounts of funds but never withdraw them.


## 1. Data Ingestion & Preprocessing


In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 1. Load the real dataset
df = pd.read_csv('../data/TS-PS4-1.csv')
print(f"Raw shape: {df.shape}")

# 2. Data Preprocessing: Handling Missing Values
# We drop rows where critical columns are entirely missing
df.dropna(subset=['amount', 'withdrawn', 'beneficiary_id'], inplace=True)

# 3. Data Preprocessing: Type Casting
# Ensure amount is numeric (coerce errors to NaN, then fill or drop)
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df['amount'].fillna(0, inplace=True) # Fill corrupted amounts with 0 safely
df['withdrawn'] = df['withdrawn'].astype(int)

print(f"Cleaned shape: {df.shape}")
print("No missing values in critical columns:\n", df[['amount', 'withdrawn', 'beneficiary_id']].isnull().sum())


Raw shape: (50000, 9)
Cleaned shape: (50000, 9)
No missing values in critical columns:
 amount            0
withdrawn         0
beneficiary_id    0
dtype: int64


## 2. Feature Engineering


In [8]:
# Feature Engineering: Group by beneficiary
features_df = df.groupby('beneficiary_id').agg(
    total_transactions=('amount', 'count'),
    total_amount=('amount', 'sum'),
    withdrawal_rate=('withdrawn', 'mean')
).reset_index()

print("\nSample engineered features before scaling:")
print(features_df.head())

# Data Preprocessing: Standard Scaling
# While Isolation Forests are scale-invariant, standardizing inputs is a crucial ML best practice
# ensuring our features are mean-centered with unit variance.
scaler = StandardScaler()
features_to_scale = ['total_transactions', 'total_amount', 'withdrawal_rate']

# We keep a copy of the unscaled data for human-readable outputs later
X_scaled = scaler.fit_transform(features_df[features_to_scale])




Sample engineered features before scaling:
  beneficiary_id  total_transactions  total_amount  withdrawal_rate
0        B100000                   1          2000              0.0
1        B100001                   1          1000              1.0
2        B100002                   1          3000              0.0
3        B100003                   1          3000              1.0
4        B100004                   1          3000              0.0


## 3. Train the Isolation Forest Model


In [ ]:
# Initialize Isolation Forest. 
# We assume approx 2% of the beneficiaries might be highly anomalous.
clf = IsolationForest(contamination=0.02, random_state=42)
clf.fit(X_scaled) # Train on SCALED data

# Predict anomalies (-1 for anomalies, 1 for normal)
features_df['anomaly_prediction'] = clf.predict(X_scaled)

# Calculate a Risk Score (0-100) based on the isolation decision function
raw_scores = clf.decision_function(X_scaled)
features_df['risk_score'] = ((raw_scores.max() - raw_scores) / (raw_scores.max() - raw_scores.min()) * 100).astype(int)

# Filter for the most severe dormant fund anomalies (high total_amount, 0 withdrawal)
flagged = features_df[features_df['withdrawal_rate'] < 0.1]
flagged = flagged.sort_values(by='risk_score', ascending=False)

print("--- Top 5 Critical Dormant Fund Anomalies Detected ---")
print(flagged.head())

--- Top 5 Critical Dormant Fund Anomalies Detected ---
      beneficiary_id  total_transactions  total_amount  withdrawal_rate  \
49999        B149999                   1          5000              0.0   
15683        B115683                   1          5000              0.0   
39400        B139400                   1          5000              0.0   
39399        B139399                   1          5000              0.0   
15672        B115672                   1          5000              0.0   

       anomaly_prediction  risk_score  
49999                   1          91  
15683                   1          91  
39400                   1          91  
39399                   1          91  
15672                   1          91  
